# T09 – Customer RFM Segmentation Tool

## MBA Project

### Table of Contents

1. [Project Overview](#1-project-overview)
2. [Project Objective](#2-project-objective)
3. [Dataset Loading](#3-dataset-loading)
4. [Data Understanding and Quality Checks](#4-data-understanding-and-quality-checks)
5. [RFM Calculation](#5-rfm-calculation)
6. [RFM Score Calculation](#6-rfm-score-calculation)
7. [Combined RFM Score](#7-combined-rfm-score)
8. [Customer Segmentation](#8-customer-segmentation)
9. [Segment Analysis](#9-segment-analysis)
10. [Campaign Strategy](#10-campaign-strategy)
11. [Customer Lookup Data and Application](#11-customer-lookup-data-and-application)
12. [Verification and Testing](#12-verification-and-testing)
13. [Business Recommendations](#13-business-recommendations)
14. [AI Use Log](#14-ai-use-log)
15. [Conclusion](#15-conclusion)


T09 – Customer RFM Segmentation Tool

## MBA Project

### Objective
To analyze customer purchasing behavior using Recency, Frequency and Monetary (RFM) analysis, classify customers into meaningful segments, and develop targeted marketing recommendations.

### Dataset
- 3,000 sales orders
- Customer transaction data
- RFM-based customer segmentation

# 1. Project Overview

This project develops a Customer RFM Segmentation Tool using sales transaction data. RFM summarizes customer behaviour using Recency, Frequency and Monetary value.

### Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Libraries imported successfully.")

Libraries imported successfully.


# 2. Project Objective

The objective is to calculate customer-level RFM measures, assign scores, create segments, design campaigns, build a customer lookup application, and verify the results.

### Upload the project workbook

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving T09_Customer_RFM_segmentation_tool.xlsx to T09_Customer_RFM_segmentation_tool.xlsx


# 3. Dataset Loading

The Sales sheet contains the transaction-level data used for the analysis.

### Read the Sales sheet

In [ ]:
file_name = "T09_Customer_RFM_segmentation_tool.xlsx"

sales = pd.read_excel(
    file_name,
    sheet_name="Sales"
)

print("Sales data loaded successfully.")
print("Rows:", sales.shape[0])
print("Columns:", sales.shape[1])

Sales data loaded successfully.
Rows: 3000
Columns: 16


# 4. Data Understanding and Quality Checks

These checks establish whether the transaction data is usable before calculating RFM.

### Preview the data

In [ ]:
sales.head()

,order_id,customer_id,order_date,region,city,state,city_tier,channel,category,product,units,unit_price_inr,discount_pct,revenue_inr,cost_inr,gross_margin_inr
0,ORD00001,CUST0040,2025-04-01,West,Surat,Gujarat,2,Marketplace,Fashion,Running Shoes,1,4949,0.0,4949.0,3773.19,1175.81
1,ORD00002,CUST0387,2025-04-01,South,Kochi,Kerala,2,Online Store,Electronics,Power Bank 20000mAh,4,1059,0.1,3812.4,2943.50,868.90
2,ORD00003,CUST0044,2025-04-01,West,Mumbai,Maharashtra,1,Online Store,Electronics,Power Bank 20000mAh,1,2379,0.1,2141.1,1317.94,823.16
3,ORD00004,CUST0067,2025-04-01,South,Hyderabad,Telangana,1,Online Store,Grocery,Cold-pressed Oil 1L,1,419,0.1,377.1,227.22,149.88
4,ORD00005,CUST0336,2025-04-01,West,Nashik,Maharashtra,2,Online Store,Beauty,Hair Oil,1,469,0.0,469.0,326.49,142.51


### Inspect structure and data types

In [ ]:
sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   order_id          3000 non-null   object        
 1   customer_id       3000 non-null   object        
 2   order_date        3000 non-null   datetime64[ns]
 3   region            3000 non-null   object        
 4   city              3000 non-null   object        
 5   state             3000 non-null   object        
 6   city_tier         3000 non-null   int64         
 7   channel           3000 non-null   object        
 8   category          3000 non-null   object        
 9   product           3000 non-null   object        
 10  units             3000 non-null   int64         
 11  unit_price_inr    3000 non-null   int64         
 12  discount_pct      3000 non-null   float64       
 13  revenue_inr       3000 non-null   float64       
 14  cost_inr          3000 n

In [ ]:
missing_values = sales.isnull().sum()

print("Missing values in each column:")
print(missing_values)


Missing values in each column:
order_id            0
customer_id         0
order_date          0
region              0
city                0
state               0
city_tier           0
channel             0
category            0
product             0
units               0
unit_price_inr      0
discount_pct        0
revenue_inr         0
cost_inr            0
gross_margin_inr    0
dtype: int64


In [ ]:
duplicate_orders = sales["order_id"].duplicated().sum()

print("Duplicate order IDs:", duplicate_orders)

Duplicate order IDs: 0


In [ ]:
unique_customers = sales["customer_id"].nunique()

print("Unique customers in Sales data:", unique_customers)

Unique customers in Sales data: 426


In [ ]:
sales["order_date"] = pd.to_datetime(
    sales["order_date"],
    errors="coerce"
)

print("Earliest order date:", sales["order_date"].min())
print("Latest order date:", sales["order_date"].max())
print("Missing/invalid dates:", sales["order_date"].isna().sum())

Earliest order date: 2025-04-01 00:00:00
Latest order date: 2026-03-31 00:00:00
Missing/invalid dates: 0


In [ ]:
print("Missing revenue values:", sales["revenue_inr"].isna().sum())
print("Zero revenue orders:", (sales["revenue_inr"] == 0).sum())
print("Negative revenue orders:", (sales["revenue_inr"] < 0).sum())

Missing revenue values: 0
Zero revenue orders: 0
Negative revenue orders: 0


In [ ]:
sales[[
    "units",
    "unit_price_inr",
    "discount_pct",
    "revenue_inr",
    "cost_inr",
    "gross_margin_inr"
]].describe()

,units,unit_price_inr,discount_pct,revenue_inr,cost_inr,gross_margin_inr
count,3000.000000,3000.000000,3000.000000,3000.000000,3000.000000,3000.000000
mean,5.677667,2225.746667,0.074567,11578.437267,7689.637683,3888.799583
std,10.674756,1939.263347,0.065850,30432.475176,20325.233771,10470.370495
min,1.000000,89.000000,0.000000,79.200000,48.190000,22.930000
25%,2.000000,769.000000,0.000000,1619.100000,1063.335000,535.760000
50%,3.000000,1709.000000,0.050000,3908.950000,2588.100000,1267.830000
75%,4.000000,3079.000000,0.100000,8775.150000,5748.340000,2994.262500
max,59.000000,9969.000000,0.250000,431528.950000,264064.980000,170593.040000


# 5. RFM Calculation

Recency is the days since the most recent purchase, Frequency is the number of orders, and Monetary is total revenue. The reference date is the latest order date.

### Calculate customer-level RFM measures

In [ ]:
reference_date = sales["order_date"].max()

rfm = sales.groupby("customer_id").agg(
    Recency=("order_date", lambda x: (reference_date - x.max()).days),
    Frequency=("order_id", "count"),
    Monetary=("revenue_inr", "sum")
).reset_index()

In [ ]:
rfm.head()

,customer_id,Recency,Frequency,Monetary
0,CUST0001,62,2,11213.25
1,CUST0002,74,3,17773.90
2,CUST0004,5,4,18771.15
3,CUST0005,16,3,10371.00
4,CUST0007,3,4,38550.85


In [ ]:
print(rfm.shape)

(426, 4)


In [ ]:
rfm[["Recency", "Frequency", "Monetary"]].describe()

,Recency,Frequency,Monetary
count,426.000000,426.000000,4.260000e+02
mean,85.861502,7.042254,8.153829e+04
std,82.697078,27.100651,3.320472e+05
min,0.000000,1.000000,4.246500e+02
25%,19.000000,2.000000,9.432675e+03
50%,62.500000,4.000000,2.638220e+04
75%,126.750000,6.000000,6.989583e+04
max,354.000000,514.000000,6.389544e+06


In [ ]:
rfm.sort_values("Frequency", ascending=False).head(10)

,customer_id,Recency,Frequency,Monetary
365,CUST0387,0,514,6389543.50
167,CUST0176,1,168,1539098.40
40,CUST0044,0,123,1250796.65
278,CUST0291,2,57,450053.45
360,CUST0382,11,47,358954.90
319,CUST0336,5,33,384104.10
375,CUST0398,10,32,462416.60
70,CUST0076,44,31,156513.65
383,CUST0406,7,31,226773.55
374,CUST0397,8,30,149919.35


In [ ]:
rfm.sort_values("Monetary", ascending=False).head(10)

,customer_id,Recency,Frequency,Monetary
365,CUST0387,0,514,6389543.50
167,CUST0176,1,168,1539098.40
40,CUST0044,0,123,1250796.65
33,CUST0036,28,26,582027.45
151,CUST0160,122,5,508752.55
375,CUST0398,10,32,462416.60
278,CUST0291,2,57,450053.45
147,CUST0156,208,5,441704.90
107,CUST0115,56,4,425924.40
149,CUST0158,5,14,386708.75


# 6. RFM Score Calculation

Customers receive scores from 1 to 5. Lower Recency days receive higher scores; higher Frequency and Monetary values receive higher scores. Rank-based quintiles are used.

### Assign R, F and M scores from 1 to 5

In [ ]:
# Create RFM scores from 1 to 5

rfm["R_Score"] = pd.qcut(
    rfm["Recency"].rank(method="first"),
    5,
    labels=[5, 4, 3, 2, 1]
).astype(int)

rfm["F_Score"] = pd.qcut(
    rfm["Frequency"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

rfm["M_Score"] = pd.qcut(
    rfm["Monetary"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

print("RFM scoring completed.")

RFM scoring completed.


In [ ]:
rfm.head(10)

,customer_id,Recency,Frequency,Monetary,R_Score,F_Score,M_Score
0,CUST0001,62,2,11213.25,3,1,2
1,CUST0002,74,3,17773.90,3,2,2
2,CUST0004,5,4,18771.15,5,3,3
3,CUST0005,16,3,10371.00,4,2,2
4,CUST0007,3,4,38550.85,5,3,4
5,CUST0008,154,1,8016.30,1,1,2
6,CUST0009,176,2,5147.10,1,1,1
7,CUST0010,29,6,334153.40,4,4,5
8,CUST0011,57,3,13119.50,3,2,2
9,CUST0012,244,2,13181.65,1,1,2


In [ ]:
print("R Score distribution:")
print(rfm["R_Score"].value_counts().sort_index())

print("\nF Score distribution:")
print(rfm["F_Score"].value_counts().sort_index())

print("\nM Score distribution:")
print(rfm["M_Score"].value_counts().sort_index())

R Score distribution:
R_Score
1    85
2    85
3    85
4    85
5    86
Name: count, dtype: int64

F Score distribution:
F_Score
1    86
2    85
3    85
4    85
5    85
Name: count, dtype: int64

M Score distribution:
M_Score
1    86
2    85
3    85
4    85
5    85
Name: count, dtype: int64


# 7. Combined RFM Score

The three individual scores are combined into a three-digit RFM score. For example, 555 means a score of 5 on all three dimensions.

### Create the combined RFM score

In [ ]:
rfm["RFM_Score"] = (
    rfm["R_Score"].astype(str)
    + rfm["F_Score"].astype(str)
    + rfm["M_Score"].astype(str)
)

rfm.head(10)

,customer_id,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score
0,CUST0001,62,2,11213.25,3,1,2,312
1,CUST0002,74,3,17773.90,3,2,2,322
2,CUST0004,5,4,18771.15,5,3,3,533
3,CUST0005,16,3,10371.00,4,2,2,422
4,CUST0007,3,4,38550.85,5,3,4,534
5,CUST0008,154,1,8016.30,1,1,2,112
6,CUST0009,176,2,5147.10,1,1,1,111
7,CUST0010,29,6,334153.40,4,4,5,445
8,CUST0011,57,3,13119.50,3,2,2,322
9,CUST0012,244,2,13181.65,1,1,2,112


# 8. Customer Segmentation

Customers are assigned using the project-defined rules in assign_segment(). These thresholds are project-specific analytical rules, not universal RFM standards.

### Apply the project segmentation rules

In [ ]:
def assign_segment(row):

    R = row["R_Score"]
    F = row["F_Score"]
    M = row["M_Score"]

    # Very recent, frequent and high-value customers
    if R >= 4 and F >= 4 and M >= 4:
        return "Champions"

    # Recent customers with reasonably strong frequency/value
    elif R >= 3 and F >= 4 and M >= 3:
        return "Loyal Customers"

    # Recent customers who have potential to become loyal
    elif R >= 4 and F >= 2 and M >= 2:
        return "Potential Loyalists"

    # Recent but low-frequency customers
    elif R >= 4 and F <= 2:
        return "New / Promising"

    # Previously valuable customers who haven't purchased recently
    elif R <= 2 and F >= 3 and M >= 3:
        return "At Risk"

    # Very inactive and relatively low-value customers
    elif R == 1 and F <= 2 and M <= 2:
        return "Lost"

    # Other inactive customers
    elif R <= 2:
        return "Hibernating"

    else:
        return "Needs Attention"


rfm["Segment"] = rfm.apply(assign_segment, axis=1)

rfm.head(10)

,customer_id,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,Segment
0,CUST0001,62,2,11213.25,3,1,2,312,Needs Attention
1,CUST0002,74,3,17773.90,3,2,2,322,Needs Attention
2,CUST0004,5,4,18771.15,5,3,3,533,Potential Loyalists
3,CUST0005,16,3,10371.00,4,2,2,422,Potential Loyalists
4,CUST0007,3,4,38550.85,5,3,4,534,Potential Loyalists
5,CUST0008,154,1,8016.30,1,1,2,112,Lost
6,CUST0009,176,2,5147.10,1,1,1,111,Lost
7,CUST0010,29,6,334153.40,4,4,5,445,Champions
8,CUST0011,57,3,13119.50,3,2,2,322,Needs Attention
9,CUST0012,244,2,13181.65,1,1,2,112,Lost


In [ ]:
segment_counts = rfm["Segment"].value_counts()

print(segment_counts)

Segment
Champions              75
Hibernating            64
Loyal Customers        55
Lost                   55
Needs Attention        54
Potential Loyalists    52
At Risk                51
New / Promising        20
Name: count, dtype: int64


# 9. Segment Analysis

The segment summary compares customer count, revenue, average RFM measures and revenue share.

### Summarize segment performance

In [ ]:
segment_summary = (
    rfm.groupby("Segment")
    .agg(
        Customers=("customer_id", "count"),
        Total_Revenue=("Monetary", "sum"),
        Avg_Recency=("Recency", "mean"),
        Avg_Frequency=("Frequency", "mean"),
        Avg_Monetary=("Monetary", "mean")
    )
    .sort_values("Total_Revenue", ascending=False)
)

segment_summary

,Customers,Total_Revenue,Avg_Recency,Avg_Frequency,Avg_Monetary
Segment,,,,,
Champions,75,20331880.60,15.920000,22.840000,271091.741333
At Risk,51,4467344.75,129.666667,5.215686,87594.995098
Loyal Customers,55,3106362.70,42.981818,7.454545,56479.321818
Hibernating,64,2447881.55,138.875000,2.468750,38248.149219
Needs Attention,54,2201941.80,62.388889,2.944444,40776.700000
Potential Loyalists,52,1552171.90,14.384615,3.384615,29849.459615
Lost,55,333074.35,237.418182,1.563636,6055.897273
New / Promising,20,294654.15,17.150000,1.600000,14732.707500


In [ ]:
segment_summary["Revenue_Share_%"] = (
    segment_summary["Total_Revenue"]
    / rfm["Monetary"].sum()
    * 100
)

segment_summary

,Customers,Total_Revenue,Avg_Recency,Avg_Frequency,Avg_Monetary,Revenue_Share_%
Segment,,,,,,
Champions,75,20331880.60,15.920000,22.840000,271091.741333,58.533750
At Risk,51,4467344.75,129.666667,5.215686,87594.995098,12.861105
Loyal Customers,55,3106362.70,42.981818,7.454545,56479.321818,8.942953
Hibernating,64,2447881.55,138.875000,2.468750,38248.149219,7.047242
Needs Attention,54,2201941.80,62.388889,2.944444,40776.700000,6.339203
Potential Loyalists,52,1552171.90,14.384615,3.384615,29849.459615,4.468571
Lost,55,333074.35,237.418182,1.563636,6055.897273,0.958893
New / Promising,20,294654.15,17.150000,1.600000,14732.707500,0.848284


In [ ]:
rfm.to_excel("RFM_Customer_Segmentation.xlsx", index=False)

print("RFM file saved successfully.")

RFM file saved successfully.


In [ ]:
segment_scores = (
    rfm.groupby("Segment")
    .agg(
        Customers=("customer_id", "count"),
        Avg_R_Score=("R_Score", "mean"),
        Avg_F_Score=("F_Score", "mean"),
        Avg_M_Score=("M_Score", "mean")
    )
    .round(2)
)

segment_scores

,Customers,Avg_R_Score,Avg_F_Score,Avg_M_Score
Segment,,,,
At Risk,51,1.80,3.76,3.82
Champions,75,4.45,4.69,4.63
Hibernating,64,1.69,2.02,2.42
Lost,55,1.00,1.29,1.25
Loyal Customers,55,3.58,4.42,3.60
Needs Attention,54,3.11,2.30,2.44
New / Promising,20,4.55,1.15,1.55
Potential Loyalists,52,4.52,2.73,2.87


In [ ]:
segment_scores = segment_scores.join(
    segment_summary[["Total_Revenue", "Revenue_Share_%"]]
)

segment_scores

,Customers,Avg_R_Score,Avg_F_Score,Avg_M_Score,Total_Revenue,Revenue_Share_%
Segment,,,,,,
At Risk,51,1.80,3.76,3.82,4467344.75,12.861105
Champions,75,4.45,4.69,4.63,20331880.60,58.533750
Hibernating,64,1.69,2.02,2.42,2447881.55,7.047242
Lost,55,1.00,1.29,1.25,333074.35,0.958893
Loyal Customers,55,3.58,4.42,3.60,3106362.70,8.942953
Needs Attention,54,3.11,2.30,2.44,2201941.80,6.339203
New / Promising,20,4.55,1.15,1.55,294654.15,0.848284
Potential Loyalists,52,4.52,2.73,2.87,1552171.90,4.468571


# 10. Campaign Strategy

Each segment is linked to a campaign objective, channel and KPI so segmentation leads to a business action.

### Define one campaign for each segment

In [ ]:
campaign_plan = pd.DataFrame({
    "Segment": [
        "Champions",
        "Loyal Customers",
        "At Risk",
        "Hibernating",
        "Needs Attention",
        "Potential Loyalists",
        "New / Promising",
        "Lost"
    ],

    "Objective": [
        "Retain high-value customers",
        "Increase customer value and retention",
        "Reactivate valuable inactive customers",
        "Re-engage inactive customers",
        "Increase purchase frequency",
        "Convert recent customers into repeat buyers",
        "Encourage first repeat purchase",
        "Attempt low-cost customer recovery"
    ],

    "Campaign": [
        "VIP Loyalty Campaign",
        "Loyalty Upgrade",
        "Win-Back Campaign",
        "Come-Back Campaign",
        "Engagement Campaign",
        "Second-Purchase Campaign",
        "Welcome-to-Loyalty Campaign",
        "Last-Chance Win-Back"
    ],

    "Channel": [
        "Email + WhatsApp",
        "Email + App/WhatsApp",
        "Email + WhatsApp",
        "Email",
        "Email + App",
        "Email + WhatsApp",
        "Email + App",
        "Email"
    ],

    "KPI": [
        "Repeat purchase rate",
        "Average order value",
        "Reactivation rate",
        "Re-engagement rate",
        "Purchase frequency",
        "Second-purchase rate",
        "Repeat purchase rate",
        "Reactivation rate"
    ]
})

campaign_plan

,Segment,Objective,Campaign,Channel,KPI
0,Champions,Retain high-value customers,VIP Loyalty Campaign,Email + WhatsApp,Repeat purchase rate
1,Loyal Customers,Increase customer value and retention,Loyalty Upgrade,Email + App/WhatsApp,Average order value
2,At Risk,Reactivate valuable inactive customers,Win-Back Campaign,Email + WhatsApp,Reactivation rate
3,Hibernating,Re-engage inactive customers,Come-Back Campaign,Email,Re-engagement rate
4,Needs Attention,Increase purchase frequency,Engagement Campaign,Email + App,Purchase frequency
5,Potential Loyalists,Convert recent customers into repeat buyers,Second-Purchase Campaign,Email + WhatsApp,Second-purchase rate
6,New / Promising,Encourage first repeat purchase,Welcome-to-Loyalty Campaign,Email + App,Repeat purchase rate
7,Lost,Attempt low-cost customer recovery,Last-Chance Win-Back,Email,Reactivation rate


In [ ]:
campaign_plan.to_excel(
    "RFM_Campaign_Plan.xlsx",
    index=False
)

print("Campaign plan saved successfully.")

Campaign plan saved successfully.


# 11. Customer Lookup Data and Application

A final customer-level CSV is created for the lookup tool. The Streamlit app lets a user enter a customer ID and view RFM details, segment and campaign.

### Create the customer-level lookup dataset

In [ ]:
rfm[
    [
        "customer_id",
        "Recency",
        "Frequency",
        "Monetary",
        "R_Score",
        "F_Score",
        "M_Score",
        "RFM_Score",
        "Segment"
    ]
].sort_values(
    ["Segment", "Monetary"],
    ascending=[True, False]
).head(20)

,customer_id,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,Segment
151,CUST0160,122,5,508752.55,2,4,5,245,At Risk
147,CUST0156,208,5,441704.90,1,4,5,145,At Risk
245,CUST0258,94,9,384316.05,2,5,5,255,At Risk
281,CUST0295,121,4,340345.50,2,3,5,235,At Risk
179,CUST0188,168,4,249357.80,1,3,5,135,At Risk
414,CUST0438,89,8,208885.65,2,5,5,255,At Risk
424,CUST0449,118,6,154676.00,2,4,5,245,At Risk
399,CUST0423,155,3,142096.35,1,3,5,135,At Risk
289,CUST0303,252,3,117579.85,1,3,5,135,At Risk
137,CUST0145,117,7,112619.80,2,4,5,245,At Risk


In [ ]:
print("Top 5 Champions:")
display(
    rfm[rfm["Segment"] == "Champions"]
    .sort_values("Monetary", ascending=False)
    .head(5)
)

print("\nTop 5 At Risk:")
display(
    rfm[rfm["Segment"] == "At Risk"]
    .sort_values("Monetary", ascending=False)
    .head(5)
)

print("\nTop 5 Lost:")
display(
    rfm[rfm["Segment"] == "Lost"]
    .sort_values("Monetary", ascending=False)
    .head(5)
)

Top 5 Champions:


,customer_id,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,Segment
365,CUST0387,0,514,6389543.50,5,5,5,555,Champions
167,CUST0176,1,168,1539098.40,5,5,5,555,Champions
40,CUST0044,0,123,1250796.65,5,5,5,555,Champions
33,CUST0036,28,26,582027.45,4,5,5,455,Champions
375,CUST0398,10,32,462416.60,5,5,5,555,Champions



Top 5 At Risk:


,customer_id,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,Segment
151,CUST0160,122,5,508752.55,2,4,5,245,At Risk
147,CUST0156,208,5,441704.90,1,4,5,145,At Risk
245,CUST0258,94,9,384316.05,2,5,5,255,At Risk
281,CUST0295,121,4,340345.50,2,3,5,235,At Risk
179,CUST0188,168,4,249357.80,1,3,5,135,At Risk



Top 5 Lost:


,customer_id,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,Segment
116,CUST0124,181,2,17215.20,1,1,2,112,Lost
24,CUST0027,208,2,16818.20,1,1,2,112,Lost
99,CUST0107,153,2,16788.40,1,1,2,112,Lost
208,CUST0219,155,1,16250.30,1,1,2,112,Lost
9,CUST0012,244,2,13181.65,1,1,2,112,Lost


In [ ]:
rfm[rfm["customer_id"] == "CUST0387"]

,customer_id,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,Segment
365,CUST0387,0,514,6389543.5,5,5,5,555,Champions


### Select final lookup columns

In [ ]:
final_rfm = rfm[
    [
        "customer_id",
        "Recency",
        "Frequency",
        "Monetary",
        "R_Score",
        "F_Score",
        "M_Score",
        "RFM_Score",
        "Segment"
    ]
].copy()

final_rfm.to_csv(
    "customer_rfm_segments.csv",
    index=False
)

print("Customer RFM file created successfully.")
print("Customers:", len(final_rfm))

Customer RFM file created successfully.
Customers: 426


### Map each segment to its recommended campaign

In [ ]:
campaign_mapping = {
    "Champions": "VIP Loyalty Campaign",
    "Loyal Customers": "Loyalty Upgrade",
    "At Risk": "Win-Back Campaign",
    "Hibernating": "Come-Back Campaign",
    "Needs Attention": "Engagement Campaign",
    "Potential Loyalists": "Second-Purchase Campaign",
    "New / Promising": "Welcome-to-Loyalty Campaign",
    "Lost": "Last-Chance Win-Back"
}

final_rfm["Recommended_Campaign"] = final_rfm["Segment"].map(campaign_mapping)

final_rfm.head()

,customer_id,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,Segment,Recommended_Campaign
0,CUST0001,62,2,11213.25,3,1,2,312,Needs Attention,Engagement Campaign
1,CUST0002,74,3,17773.90,3,2,2,322,Needs Attention,Engagement Campaign
2,CUST0004,5,4,18771.15,5,3,3,533,Potential Loyalists,Second-Purchase Campaign
3,CUST0005,16,3,10371.00,4,2,2,422,Potential Loyalists,Second-Purchase Campaign
4,CUST0007,3,4,38550.85,5,3,4,534,Potential Loyalists,Second-Purchase Campaign


### Save the lookup file used by the app

In [ ]:
final_rfm.to_csv(
    "customer_rfm_segments.csv",
    index=False
)

print("Final customer lookup file saved.")

Final customer lookup file saved.


In [ ]:
customer_id = "CUST0387"

customer_result = final_rfm[
    final_rfm["customer_id"] == customer_id
]

customer_result

,customer_id,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,Segment,Recommended_Campaign
365,CUST0387,0,514,6389543.5,5,5,5,555,Champions,VIP Loyalty Campaign


### Streamlit application code

The application code is written to `app.py` and deployed separately from this analysis notebook.

In [ ]:
%%writefile app.py

import streamlit as st
import pandas as pd

# -----------------------------
# Page configuration
# -----------------------------
st.set_page_config(
    page_title="Customer RFM Segmentation",
    page_icon="📊",
    layout="centered"
)

# -----------------------------
# Load RFM data
# -----------------------------
rfm = pd.read_csv("customer_rfm_segments.csv")

# -----------------------------
# Title
# -----------------------------
st.title("📊 Customer RFM Segmentation Tool")

st.write(
    "Enter a customer ID to view their RFM profile, "
    "customer segment and recommended campaign."
)

# -----------------------------
# Customer lookup
# -----------------------------
customer_id = st.text_input(
    "Enter Customer ID",
    placeholder="Example: CUST0387"
).strip().upper()

# -----------------------------
# Lookup button
# -----------------------------
if st.button("🔍 Look Up Customer"):

    if customer_id == "":
        st.warning("Please enter a customer ID.")

    else:
        customer = rfm[
            rfm["customer_id"] == customer_id
        ]

        if customer.empty:
            st.error(
                "Customer ID not found. Please check the ID and try again."
            )

        else:
            customer = customer.iloc[0]

            st.success(
                f"Customer found: {customer['customer_id']}"
            )

            # -----------------------------
            # Customer segment
            # -----------------------------
            st.subheader("Customer Segment")

            st.info(
                f"### {customer['Segment']}"
            )

            # -----------------------------
            # RFM information
            # -----------------------------
            st.subheader("RFM Profile")

            col1, col2, col3 = st.columns(3)

            with col1:
                st.metric(
                    "Recency",
                    f"{int(customer['Recency'])} days"
                )

            with col2:
                st.metric(
                    "Frequency",
                    f"{int(customer['Frequency'])} orders"
                )

            with col3:
                st.metric(
                    "Monetary",
                    f"₹{customer['Monetary']:,.2f}"
                )

            # -----------------------------
            # RFM scores
            # -----------------------------
            st.subheader("RFM Scores")

            st.write(
                f"**R Score:** {customer['R_Score']}  |  "
                f"**F Score:** {customer['F_Score']}  |  "
                f"**M Score:** {customer['M_Score']}"
            )

            st.write(
                f"**Combined RFM Score:** "
                f"{customer['RFM_Score']}"
            )

            # -----------------------------
            # Campaign
            # -----------------------------
            st.subheader("Recommended Campaign")

            st.success(
                customer["Recommended_Campaign"]
            )

# -----------------------------
# Footer
# -----------------------------
st.divider()

st.caption(
    "RFM = Recency, Frequency and Monetary Value. "
    "Segments are generated using quintile-based RFM scoring "
    "and rule-based customer segmentation."
)

Writing app.py


# 12. Verification and Testing

Verification checks completeness, selected customer calculations and invalid customer handling.

### Automated verification

In [ ]:
print("Total customers:", len(rfm))
print("Missing segments:", rfm["Segment"].isna().sum())
print("Missing R scores:", rfm["R_Score"].isna().sum())
print("Missing F scores:", rfm["F_Score"].isna().sum())
print("Missing M scores:", rfm["M_Score"].isna().sum())

print("\nSegment counts:")
print(rfm["Segment"].value_counts())

Total customers: 426
Missing segments: 0
Missing R scores: 0
Missing F scores: 0
Missing M scores: 0

Segment counts:
Segment
Champions              75
Hibernating            64
Loyal Customers        55
Lost                   55
Needs Attention        54
Potential Loyalists    52
At Risk                51
New / Promising        20
Name: count, dtype: int64


### Verify a customer against the original Sales transactions

In [ ]:
test_customer = "CUST0387"

customer_orders = sales[sales["customer_id"] == test_customer]

print("Customer:", test_customer)
print("Number of orders:", len(customer_orders))
print("Latest order:", customer_orders["order_date"].max())
print("Total revenue:", customer_orders["revenue_inr"].sum())

Customer: CUST0387
Number of orders: 514
Latest order: 2026-03-31 00:00:00
Total revenue: 6389543.5


### Verify multiple customer records

In [ ]:
verification_customers = ["CUST0387", "CUST0176", "CUST0044"]

verification_table = rfm[
    rfm["customer_id"].isin(verification_customers)
][[
    "customer_id",
    "Recency",
    "Frequency",
    "Monetary",
    "R_Score",
    "F_Score",
    "M_Score",
    "RFM_Score",
    "Segment"
]]

verification_table

,customer_id,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,Segment
40,CUST0044,0,123,1250796.65,5,5,5,555,Champions
167,CUST0176,1,168,1539098.40,5,5,5,555,Champions
365,CUST0387,0,514,6389543.50,5,5,5,555,Champions


# 13. Business Recommendations

Recommendations should be based on the actual segment characteristics and revenue contribution observed in the analysis.

### Segment-level actions

- **Champions:** retention and loyalty benefits.
- **Loyal Customers:** encourage continued repeat purchasing and higher-value purchases.
- **At Risk:** targeted win-back communication.
- **Hibernating:** re-engagement with relevant recommendations or offers.
- **Needs Attention:** increase engagement and purchase frequency.
- **Potential Loyalists:** encourage repeat purchases and stronger loyalty.
- **New / Promising:** encourage the next purchase and early loyalty.
- **Lost:** targeted, low-cost recovery attempt.


### Final segment summary for reporting

In [ ]:
segment_summary = rfm.groupby("Segment").agg(
    Customers=("customer_id", "count"),
    Total_Revenue=("Monetary", "sum"),
    Avg_Recency=("Recency", "mean"),
    Avg_Frequency=("Frequency", "mean"),
    Avg_Monetary=("Monetary", "mean")
).reset_index()

total_revenue = rfm["Monetary"].sum()

segment_summary["Revenue_Share_%"] = (
    segment_summary["Total_Revenue"] / total_revenue * 100
)

segment_summary

,Segment,Customers,Total_Revenue,Avg_Recency,Avg_Frequency,Avg_Monetary,Revenue_Share_%
0,At Risk,51,4467344.75,129.666667,5.215686,87594.995098,12.861105
1,Champions,75,20331880.60,15.920000,22.840000,271091.741333,58.533750
2,Hibernating,64,2447881.55,138.875000,2.468750,38248.149219,7.047242
3,Lost,55,333074.35,237.418182,1.563636,6055.897273,0.958893
4,Loyal Customers,55,3106362.70,42.981818,7.454545,56479.321818,8.942953
5,Needs Attention,54,2201941.80,62.388889,2.944444,40776.700000,6.339203
6,New / Promising,20,294654.15,17.150000,1.600000,14732.707500,0.848284
7,Potential Loyalists,52,1552171.90,14.384615,3.384615,29849.459615,4.468571


# 14. AI Use Log

Record only the AI assistance actually used during the project. Do not claim prompts or outputs that were not used.

| Step | AI use | Prompt / purpose | Output used? | Human verification / modification |
|---|---|---|---|---|
| 1 | | | | |
| 2 | | | | |
| 3 | | | | |
| 4 | | | | |
| 5 | | | | |

**Fill this table with the actual AI interactions used during the project.**

### Campaign plan output

In [ ]:
campaign_plan

,Segment,Objective,Campaign,Channel,KPI
0,Champions,Retain high-value customers,VIP Loyalty Campaign,Email + WhatsApp,Repeat purchase rate
1,Loyal Customers,Increase customer value and retention,Loyalty Upgrade,Email + App/WhatsApp,Average order value
2,At Risk,Reactivate valuable inactive customers,Win-Back Campaign,Email + WhatsApp,Reactivation rate
3,Hibernating,Re-engage inactive customers,Come-Back Campaign,Email,Re-engagement rate
4,Needs Attention,Increase purchase frequency,Engagement Campaign,Email + App,Purchase frequency
5,Potential Loyalists,Convert recent customers into repeat buyers,Second-Purchase Campaign,Email + WhatsApp,Second-purchase rate
6,New / Promising,Encourage first repeat purchase,Welcome-to-Loyalty Campaign,Email + App,Repeat purchase rate
7,Lost,Attempt low-cost customer recovery,Last-Chance Win-Back,Email,Reactivation rate


# 15. Conclusion

The project produces customer-level RFM measures, five-point R/F/M scores, rule-based customer segments, segment-specific campaigns and a Streamlit lookup application. The final verification cells provide checks on completeness and selected customer calculations.